# **Middlewares with LangChain Agent**

## **What's Covered?**
1. Introduction to Middleware
    - What is Middleware?
    - What You Can Do With Middleware?
    - How Middleware Work?
2. PII Middleware
3. Summarization Middleware
4. Model Call Limit Middleware
5. Tool Run Limit Middleware
6. Human In The Loop Middleware

## **Introduction to Middleware**

### **What is Middleware?**
Middleware is a feature in LangChain that lets you step inside an agent’s workflow and make changes before, during, or after the model does something.

Think of it like a checkpoint system where you can plug in extra logic — without modifying the main agent code.

### **What You Can Do With Middleware?**
Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

### **How Middleware Work?**
The core agent loop involves:
1. calling a model,
2. letting it choose tools to execute, and
3. then finishing when it calls no more tools

Middleware exposes hooks before and after each of those steps.
<div style="display:flex; gap:20px;">
    <img src="assets/agent_loop.png" style="width:40%; height:auto;">
    <img src="assets/agent_loop_with_middlewares.png" style="width:40%; height:auto;">
</div>

## **PII Middleware**
Detect and handle Personally Identifiable Information (PII) in conversations. This middleware detects common PII types and applies configurable strategies to handle them. It can detect emails, credit cards, IP addresses, MAC addresses, and URLs in both user input and agent output.

**Configuration options:**
- pii_type: `Literal['email', 'credit_card', 'ip', 'mac_address', 'url']` | `str`
- strategy:
    - block: Raise an exception when PII is detected
    - redact: Replace PII with `[REDACTED_TYPE]` placeholders
    - mask: Partially mask PII (e.g., `****-****-****-1234` for credit card)
    - hash: Replace PII with deterministic hash (e.g., `<email_hash:a1b2c3d4>`)
- apply_to_input: bool = True
- apply_to_output: bool = False
- apply_to_tool_results: bool = False

In [11]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

agent = create_agent(
    model=chat_model,
    tools=[],
    middleware=[
        PIIMiddleware(pii_type="email", strategy="redact", apply_to_input=True, apply_to_output=True),
        PIIMiddleware(pii_type="credit_card", strategy="mask", apply_to_input=True, apply_to_output=True),
    ],
)

response = agent.invoke(
    {
        "messages": "generate a markdown table with 5 random datapoints with features like name, email, ip addresses and credit card number"
    }
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

generate a markdown table with 5 random datapoints with features like name, email, ip addresses and credit card number
================================== Ai Message ==================================

I can generate a Markdown table with 5 random datapoints, but keep in mind that generating real credit card numbers is difficult due to the strict regulations around their use and format. I will use a placeholder in the format "XXXX-XXXX-XXXX-XXXX" for the credit card number, as actual credit card data is sensitive and should never be shared lightly.

### Random Datapoints Table
| Name        | Email                | IP Address       | Credit Card   | Country  |
|-------------|----------------------|------------------|----------------|----------|
| Emily Chen  | [REDACTED_EMAIL]    | 192.168.1.100    | 1234-5678-9012-3456  | USA      |
| Ethan Lee   | [REDACTED_EMAIL]     | 10.0.0.1         | 5678-9012-3456-

## **Summarization Middleware**

The summarization middleware monitors message token counts and automatically summarizes older messages when thresholds are reached.

**Configuration options:**
- model: The language model to use for generating summaries.
- summary_prompt: Prompt template for generating summaries. There exist a default prompt template.
- trigger: One or more thresholds that trigger summarization.
    - `("messages", 50)`: Trigger summarization when 50 messages is reached
    - `("tokens", 3000)`: Trigger summarization when 3000 tokens is reached
    - `[("fraction", 0.8), ("messages", 100)]`: Trigger summarization either when 80% of model's max input tokens is reached or when 100 messages is reached (whichever comes first)
- keep: Context retention policy applied after summarization.
    - `("messages", 20)`: Keep the most recent 20 messages
    - `("tokens", 3000)`: Keep the most recent 3000 tokens
    - `("fraction", 0.3)`: Keep the most recent 30% of the model's max input tokens
- trim_tokens_to_summarize: Maximum tokens to keep when preparing messages for the summarization call. Pass `None` to skip trimming entirely. Default to `4000` tokens.

In [3]:
# ! pip install wikipedia langchain-community

In [4]:
from langchain_community.retrievers import WikipediaRetriever

retriever = WikipediaRetriever(
    top_k_results=1,
    doc_content_chars_max=20_000,
)

In [5]:
from langchain_core.tools import tool

@tool
def fetch_wikipedia_data(query: str) -> str:
    """Fetch content of Wikipedia page from top hit of a query."""
    results = retriever.invoke(query)
    if results:
        return results[0].page_content
    return "(No data found)"

In [8]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-20b", 
    temperature=1
)

summarization_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [6]:
summary_prompt = """
Summarize the main thrust of this conversation. What have the human and assistant
discussed so far? Focus on key facts and requests.
<messages>
Messages to summarize:
{messages}
</messages>
"""

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=chat_model,
    tools=[fetch_wikipedia_data],
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            summary_prompt=summary_prompt,
            # Trigger summarization when 70% of context is used
            trigger=("fraction", 0.7),
            # Keep the most recent 30% of messages in full
            keep=("fraction", 0.3),
            # No additional trimming before summarization
            trim_tokens_to_summarize=None,
        ),
    ],
)

## **Model Call Limit Middleware**

Limit the number of model calls to prevent infinite loops or excessive costs. 

Model call limit is useful for the following:
- Preventing runaway agents from making too many API calls.
- Enforcing cost controls on production deployments.
- Testing agent behavior within specific call budgets.

This middleware monitors the number of model calls made during agent execution and can terminate the agent when specified limits are reached. It supports both **thread-level** and **run-level** call counting with configurable exit behaviors.
- **Thread-level (i.e. Conversation Level)**: The middleware tracks the number of model calls and persists call count across multiple runs (invocations) of the agent.
- **Run-level (i.e. Single Invocation)**: The middleware tracks the number of model calls made during a single run (invocation) of the agent.

**Configuration options:**
- thread_limit: Maximum model calls across all runs in a thread. Defaults to no limit. For thread_limit, it is mandatory to provide a `checkpointer`
- run_limit: Maximum model calls per single invocation. Defaults to no limit.
- exit_behavior: Behavior when limit is reached. Options: `end` (graceful termination) or `error` (raise exception). Default to `end`.

In [1]:
from langchain_core.tools import tool

@tool
def multiply_tool(a: int, b: int) -> int:
    """This tool take two integer variables in the input and returns the product"""
    return a * b

In [2]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

# Create middleware with limits
call_tracker = ModelCallLimitMiddleware(run_limit=2, exit_behavior="end")

agent = create_agent(
    model=chat_model,
    tools=[multiply_tool],
    middleware=[call_tracker]
)

In [4]:
response = agent.invoke({"messages": "help me multiply 3 and 4"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

help me multiply 3 and 4
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_16414aeb-747e-4167-821a-9d18012409a4)
 Call ID: fc_16414aeb-747e-4167-821a-9d18012409a4
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiply_tool

12
================================== Ai Message ==================================

Sure! 3 × 4 = 12.


In [11]:
response = agent.invoke({"messages": "help me multiply 3 and 4 and 5"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

help me multiply 3 and 4 and 5
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_8fb209ff-321f-443f-a56f-6b99a57dd804)
 Call ID: fc_8fb209ff-321f-443f-a56f-6b99a57dd804
  Args:
    a: 3
    b: 4
================================= Tool Message =================================
Name: multiply_tool

12
================================== Ai Message ==================================
Tool Calls:
  multiply_tool (fc_b435fb1e-9bc8-423a-a863-9156d583acff)
 Call ID: fc_b435fb1e-9bc8-423a-a863-9156d583acff
  Args:
    a: 12
    b: 5
================================= Tool Message =================================
Name: multiply_tool

60
================================== Ai Message ==================================

Model call limits exceeded: run limit (2/2)


## **Tool Run Limit Middleware**

- tool_name: str | None = None
- thread_limit: int | None = None
- run_limit: int | None = None
- exit_behavior:
    - 'continue': Block exceeded tools, let execution continue (default), agent resumes
    - 'error': raise informative error message
    - 'end': Stop immediately with a ToolMessage + AI message for the single tool call that exceeded the limit (raises NotImplementedError if there are other pending tool calls (due to parallel tool calling).

In [5]:
from langchain_core.tools import tool

@tool(parse_docstring=True)
def search_items(query: str) -> str:
    """Search for items in the store.
    
    Args:
        query: Search query to find items.
        
    Returns:
        List of matching items with IDs and prices.
    """
    return """Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5"""


@tool(parse_docstring=True)
def purchase_item(item_id: str) -> str:
    """Purchase an item by its ID.
    
    Args:
        item_id: The unique identifier of the item to purchase.
        
    Returns:
        Purchase confirmation.
    """
    # Simulate purchase
    return f"✓ Successfully purchased item {item_id}"

In [6]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)

chat_model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware

PROMPT = """
You are a shopping assistant. Help users search for and purchase items.
Use the search_items tool to find items and the purchase_item tool to purchase items.
Make sure to use the ratings of the items to make the best purchase decision.
"""

agent = create_agent(
    model=chat_model,
    tools=[search_items, purchase_item],
    system_prompt=PROMPT,
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="purchase_item",
            run_limit=1,  # Max 1 purchase per conversation turn
            thread_limit=2,  # Max 2 purchases total across all conversation turns
        )
    ],
)

In [8]:
response = agent.invoke({"messages": "find the headphones under 300$"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

find the headphones under 300$
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_faf716bc-7c32-4530-959a-f43c19ea9339)
 Call ID: fc_faf716bc-7c32-4530-959a-f43c19ea9339
  Args:
    query: headphones under $300
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================

Here are the headphones under $300:

| ID | Product | Price | Rating |
|----|---------|-------|--------|
| **hdpn-002** | Bose QuietComfort 45 | **$279.00** | **4.7 / 5** |
| hdpn-001 | Sony WH‑1000XM5 | $299.99 | 4.5 / 5 |

The Bose QuietComfort 45 has the higher

In [9]:
response = agent.invoke({"messages": "purchase a Bose QuietComfort 45"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

purchase a Bose QuietComfort 45
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_3e437333-3d77-43b0-86eb-9cba62f8c285)
 Call ID: fc_3e437333-3d77-43b0-86eb-9cba62f8c285
  Args:
    query: Bose QuietComfort 45
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_cf2b092c-6cab-4081-b324-a791f3ca5992)
 Call ID: fc_cf2b092c-6cab-4081-b324-a791f3ca5992
  Args:
    item_id: hdpn-002
================================= Tool Message =================================
Name: purchase_item

✓ Successfull

In [10]:
response = agent.invoke({"messages": "purchase me 2 pairs of Bose QuietComfort 45"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

purchase me 2 pairs of Bose QuietComfort 45
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_3f36601e-27d9-466e-8925-157b199509da)
 Call ID: fc_3f36601e-27d9-466e-8925-157b199509da
  Args:
    query: Bose QuietComfort 45
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_87df53b3-f5ef-48da-9588-78f07a7a87d7)
 Call ID: fc_87df53b3-f5ef-48da-9588-78f07a7a87d7
  Args:
    item_id: hdpn-002
================================= Tool Message =================================
Name: purchase_item

✓

In [12]:
response = agent.invoke({"messages": "purchase me two best pairs of headphones under 300$"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

purchase me two best pairs of headphones under 300$
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_e9a9804f-d19b-4f73-bd3f-221b38fbd9b7)
 Call ID: fc_e9a9804f-d19b-4f73-bd3f-221b38fbd9b7
  Args:
    query: headphones under $300
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_58490003-ed71-42df-8f7d-9c12c2b3110e)
 Call ID: fc_58490003-ed71-42df-8f7d-9c12c2b3110e
  Args:
    item_id: hdpn-002
================================= Tool Message =================================
Name: purchas

## **Human In The Loop Middleware**

- interrupt_on: `dict[str, bool | InterruptOnConfig]` - Mapping of tool name to allowed actions. If a tool doesn't have an entry, it's auto-approved by default.
- description_prefix: `str` - The prefix to use when constructing action requests. Not used if a tool has a `description` in its `InterruptOnConfig`.


InterruptOnConfig defines configuration for an action requiring human in the loop. It carries following args:
- allowed_decisions: `DecisionType = Literal['approve', 'edit', 'reject']`
- description: `str`
- args_schema: `NotRequired[dict[str, Any]]` - JSON schema for the args associated with the action, if edits are allowed.

In [13]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model=chat_model,
    tools=[search_items, purchase_item],
    system_prompt=PROMPT,
    middleware=[
        ToolCallLimitMiddleware(
            tool_name="purchase_item",
            run_limit=1,  # Max 1 purchase per conversation turn
            thread_limit=2,  # Max 2 purchases total across all conversation turns
        ), 
        HumanInTheLoopMiddleware(
            interrupt_on={"purchase_item": True}
        )
    ],
)

In [14]:
response = agent.invoke({"messages": "find the headphones under 300$"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

find the headphones under 300$
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_f9841677-bef5-4386-99da-5bcbe494be53)
 Call ID: fc_f9841677-bef5-4386-99da-5bcbe494be53
  Args:
    query: headphones under $300
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================

Here are the headphones that fit your budget (under $300):

| ID | Model | Price | Rating |
|----|-------|-------|--------|
| **hdpn-002** | Bose QuietComfort 45 | **$279.00** | **4.7 / 5** |
| hdpn-001 | Sony WH‑1000XM5 | $299.99 | 4.5 / 5 |

The Bose QuietComfor

In [15]:
response = agent.invoke({"messages": "yes, please purchase Bose QuietComfort 45"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

yes, please purchase Bose QuietComfort 45
================================== Ai Message ==================================
Tool Calls:
  search_items (fc_f86431cc-eee8-4bd7-a34c-6351bf9b9d5d)
 Call ID: fc_f86431cc-eee8-4bd7-a34c-6351bf9b9d5d
  Args:
    query: Bose QuietComfort 45
================================= Tool Message =================================
Name: search_items

Found 3 items:
    - ID: hdpn-001, Sony WH-1000XM5, $299.99, rating 4.5/5
    - ID: hdpn-002, Bose QuietComfort 45, $279.00, rating 4.7/5
    - ID: hdpn-003, Apple AirPods Max, $549.00, rating 4.8/5
================================== Ai Message ==================================
Tool Calls:
  purchase_item (fc_c5c5a048-2f3c-4783-b439-0e95b3dd4dd6)
 Call ID: fc_c5c5a048-2f3c-4783-b439-0e95b3dd4dd6
  Args:
    item_id: hdpn-002


In [16]:
response

{'messages': [HumanMessage(content='yes, please purchase Bose QuietComfort\u202f45', additional_kwargs={}, response_metadata={}, id='2e90a381-801d-4d88-87a2-f427fc96734c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to purchase Bose QuietComfort 45. First search for items.', 'tool_calls': [{'id': 'fc_f86431cc-eee8-4bd7-a34c-6351bf9b9d5d', 'function': {'arguments': '{"query":"Bose QuietComfort 45"}', 'name': 'search_items'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 217, 'total_tokens': 265, 'completion_time': 0.10735181, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.008752131, 'prompt_tokens_details': None, 'queue_time': 0.051258188, 'total_time': 0.116103941}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d81b3304b3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dab86-cab6-7861-b0a4